In [ ]:
import os
from hydra import initialize, compose
from omegaconf import OmegaConf
from pathlib import Path
from loguru import logger
import pandas as pd

In [ ]:
hydra_config_path: Path = Path("workspace_dani/conf")
hydra_config_path_relative = hydra_config_path.relative_to(".")

In [ ]:
with initialize(version_base=None, config_path=str("../../conf"), job_name="config_table_generation"):
    # cfg = compose(overrides=["+table=apache_patient_results"])
    cfg = compose(config_name="config_table_generation.yaml", overrides=["table=apache_aps_var"], return_hydra_config=True)
    print(cfg)

In [ ]:
cwd_path: Path = Path.cwd()
project_root_path: Path = cwd_path.parents[2]
parquet_data_path: Path = project_root_path / Path(cfg.table.paths.table_location)
print(cwd_path)
print(project_root_path)
print(parquet_data_path) 

In [ ]:
def load_parquet_table(parquet_path):
    
    logger.info(f"Trying to load from {parquet_data_path = }")
    
    if parquet_path.is_file():

        df_table = pd.read_parquet(
            parquet_path,
            engine="pyarrow",
        # use_nullable_dtypes=True,
            # dtype_backend="pyarrow"
        )

        logger.debug(f"Summary of the dataframe: \n {df_table}")
        logger.info(f"Data types of the dataframe: \n {df_table.info()}")

    else:
        logger.error(f"Parquet File not found")
        df_table = None
    
    return df_table

In [ ]:
df_data = load_parquet_table(parquet_path=parquet_data_path)

In [ ]:
def load_excluded_patient_unit_stay_ids(ids_path: Path): 
    df_excluded_ids: pd.DataFrame = pd.read_parquet(path=ids_path)
    excluded_id_list: list = df_excluded_ids["Excluded ID"].to_list()
    return excluded_id_list 

In [ ]:
excluded_ids_path = project_root_path / Path("intermediary_data/patient_unit_stay_ids/exclusion/admit_source_unit_stay/excluded_ids_admit_source_unit_type.parquet")

In [ ]:
def remove_excluded_ids_from_df(df_input: pd.DataFrame, excluded_ids:list):
    df_removed = df_input[~df_input["patientunitstayid"].isin(excluded_ids)]
    return df_removed


In [ ]:
drop_ids = load_excluded_patient_unit_stay_ids(ids_path=excluded_ids_path)


In [ ]:
df_leftover = df_data[~df_data["patientunitstayid"].isin(drop_ids)]

In [ ]:
df_leftover.head()